# Improving the Model — Augmentation + 5-Seed Ensemble

This notebook tries to beat the baseline from `colab_train_eval.ipynb`
(**ROC-AUC 0.9017**) using two changes that do not need any new data:

1. **Data augmentation** — make slightly altered copies of the training signals
   every epoch, so the model sees more variety than the 11,622 recordings alone.
2. **A 5-model ensemble** — train the same network 5 times from different random
   starts and average their answers. Mistakes tend to be random, so averaging
   cancels some of them out.

The task itself is unchanged: **clean Normal (9,069) vs MI (5,469), Lead I, 100 Hz**,
using the official PTB-XL folds (1–8 train, 9 validation, 10 test).

### A note on "accuracy"

We tune for **ROC-AUC**, not accuracy. Accuracy is misleading here: answering
"Normal" to everything already scores 62.4% on the test fold, and the operating
point that misses the fewest heart attacks actually has the *lowest* accuracy of
the three. AUC measures how well the two classes separate, independent of where
the threshold is drawn — improve it and every threshold improves at once.

The final table reports accuracy anyway, so you have it for the thesis.

**Runtime:** ~25–35 minutes on a T4 GPU. Runtime → Change runtime type → T4 GPU.

## 1. Setup

In [ ]:
import os, json, numpy as np, tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus[0].name if gpus else "NONE - this will be slow, switch to T4")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/ecg_project"

DATA_PATH    = os.path.join(DRIVE_DIR, "ptbxl_leadI.npz")
OUT_DIR      = os.path.join(DRIVE_DIR, "outputs")
ENSEMBLE_DIR = os.path.join(OUT_DIR, "ensemble")
os.makedirs(ENSEMBLE_DIR, exist_ok=True)

BASELINE_PATH = os.path.join(OUT_DIR, "mi_cnn_model.keras")   # from notebook 1

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Missing {DATA_PATH} - upload ptbxl_leadI.npz first.")

print("data     :", DATA_PATH)
print("ensemble :", ENSEMBLE_DIR)
print("baseline :", BASELINE_PATH, "(found)" if os.path.exists(BASELINE_PATH) else "(NOT found - comparison will be skipped)")

## 2. Load and split (identical to the baseline notebook)

In [ ]:
bundle = np.load(DATA_PATH)
X_raw, y, folds = bundle["X"], bundle["y"], bundle["folds"]

# Per-recording z-score, exactly as before.
X = (X_raw - X_raw.mean(axis=1, keepdims=True)) / (X_raw.std(axis=1, keepdims=True) + 1e-8)
X = X[..., np.newaxis].astype(np.float32)

train_mask = np.isin(folds, [1, 2, 3, 4, 5, 6, 7, 8])
val_mask, test_mask = folds == 9, folds == 10

X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f"train {len(y_train):5d}   val {len(y_val):4d}   test {len(y_test):4d}")

# Class weights -> per-sample weights, because tf.data pipelines take
# sample_weight rather than the class_weight argument.
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
w_train = np.where(y_train == 1, cw[1], cw[0]).astype(np.float32)
print("class weights:", {0: round(float(cw[0]), 3), 1: round(float(cw[1]), 3)})

## 3. Data augmentation

Every epoch each training signal is altered slightly, so the model never sees
exactly the same input twice. This fights over-fitting, and — just as useful for
this project — it teaches the model to cope with the kind of mess a real AD8232
electrode produces, which supports **RQ1**.

Three augmentations, each chosen because it mimics something real:

| Augmentation | Mimics | Setting |
|---|---|---|
| **Time shift** | The heartbeat not starting neatly at t = 0 | ±1 second |
| **Baseline wander** | Chest movement and breathing pulling the trace up and down | 0.15–0.5 Hz, amplitude ≤ 0.15 |
| **Gaussian noise** | Electrical noise from cheap electrodes and wiring | σ up to 0.05 |

The shift uses **reflect padding** rather than wrapping the signal around, so no
artificial jump is created at the join.

**Why there is no amplitude scaling.** It would be a no-op. Our preprocessing
z-scores the signal *last*, and scaling a signal before z-scoring gives back
exactly the same result — the normalisation cancels it. Z-scoring already makes
the model immune to gain differences, so that robustness is free.

Each augmented signal is **re-z-scored at the end**, because that is the order
the live device will use: whatever arrives from the ESP32 gets normalised last.

In [ ]:
SIG_LEN = X.shape[1]      # 1000
PAD = 100                 # +/- 1 second of shift at 100 Hz
TWO_PI = 2.0 * np.pi


def augment(x, y, w):
    """Randomly alter ONE training signal. Runs on the GPU inside tf.data."""
    # --- 1. random time shift (reflect-pad then crop, so no artificial jump) ---
    xp = tf.pad(x, [[PAD, PAD], [0, 0]], mode="REFLECT")
    off = tf.random.uniform([], 0, 2 * PAD + 1, dtype=tf.int32)
    x = tf.slice(xp, [off, 0], [SIG_LEN, 1])

    # --- 2. baseline wander: a slow drift, like breathing ---
    t = tf.linspace(0.0, 10.0, SIG_LEN)[:, None]          # seconds
    amp = tf.random.uniform([], 0.0, 0.15)
    frq = tf.random.uniform([], 0.15, 0.5)                # Hz
    pha = tf.random.uniform([], 0.0, TWO_PI)
    x = x + amp * tf.sin(TWO_PI * frq * t + pha)

    # --- 3. electrical noise ---
    sd = tf.random.uniform([], 0.0, 0.05)
    x = x + tf.random.normal(tf.shape(x), stddev=sd)

    # --- 4. re-normalise, because the live pipeline z-scores last ---
    x = (x - tf.reduce_mean(x)) / (tf.math.reduce_std(x) + 1e-8)
    return x, y, w


BATCH = 64

def make_train_ds(seed):
    ds = tf.data.Dataset.from_tensor_slices((X_train, y_train, w_train))
    ds = ds.shuffle(len(y_train), seed=seed, reshuffle_each_iteration=True)
    ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

# Validation and test are NEVER augmented - they must stay realistic.
val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(256)
print("augmentation pipeline ready")

### Check the augmentation looks sensible

Never trust an augmentation you have not looked at. The grey line is the original
recording; the coloured lines are three augmented versions. They should look like
the *same heartbeat* recorded on a worse day — if the shape is destroyed, the
model is being taught nonsense.

In [ ]:
sample = X_train[np.where(y_train == 1)[0][0]]         # first MI recording
t_axis = np.arange(SIG_LEN) / 100.0

plt.figure(figsize=(14, 6))
for k in range(3):
    aug, _, _ = augment(tf.constant(sample), tf.constant(1), tf.constant(1.0))
    plt.subplot(3, 1, k + 1)
    plt.plot(t_axis, sample.ravel(), color="#bbb", lw=1.6, label="original")
    plt.plot(t_axis, aug.numpy().ravel(), lw=0.9, label=f"augmented #{k+1}")
    plt.legend(loc="upper right", fontsize=8); plt.ylabel("z-score")
plt.xlabel("seconds")
plt.suptitle("Augmentation check - shape must survive", y=0.995)
plt.tight_layout(); plt.show()

## 4. The model (unchanged from the baseline)

In [ ]:
from tensorflow.keras import layers, models, callbacks


def conv_block(x, filters, kernel_size, dropout):
    for _ in range(2):
        x = layers.Conv1D(filters, kernel_size, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
    x = layers.MaxPooling1D(2)(x)
    return layers.Dropout(dropout)(x)


def build_model():
    inp = layers.Input(shape=(SIG_LEN, 1))
    x = conv_block(inp, 32, 7, 0.10)
    x = conv_block(x,   64, 5, 0.10)
    x = conv_block(x,  128, 3, 0.20)
    x = conv_block(x,  128, 3, 0.20)
    x = layers.Concatenate()([layers.GlobalAveragePooling1D()(x),
                              layers.GlobalMaxPooling1D()(x)])
    x = layers.Dropout(0.5)(layers.Dense(64, activation="relu")(x))
    out = layers.Dense(1, activation="sigmoid")(x)

    m = models.Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy",
              metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return m


print("parameters:", f"{build_model().count_params():,}")

## 5. Train 5 models

Same architecture, same data, five different random starting points. Because
augmentation slows over-fitting down, each model is allowed to train longer than
the baseline did (80 epochs, patience 12).

Each finished model is saved to Drive, so if Colab disconnects part-way you keep
the models already trained.

In [ ]:
SEEDS = [42, 43, 44, 45, 46]
val_probs_all, test_probs_all, histories = [], [], []

for n, seed in enumerate(SEEDS, 1):
    print(f"\n{'=' * 60}\n  MODEL {n} of {len(SEEDS)}  (seed {seed})\n{'=' * 60}")
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)

    path = os.path.join(ENSEMBLE_DIR, f"model_seed{seed}.keras")
    model = build_model()

    hist = model.fit(
        make_train_ds(seed),
        validation_data=val_ds,
        epochs=80,
        callbacks=[
            callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=12,
                                    restore_best_weights=True, verbose=1),
            callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5,
                                        patience=5, min_lr=1e-5, verbose=0),
        ],
        verbose=2,
    )

    model.save(path)
    val_probs_all.append(model.predict(X_val, verbose=0).ravel())
    test_probs_all.append(model.predict(X_test, verbose=0).ravel())
    histories.append(hist.history)

    from sklearn.metrics import roc_auc_score
    print(f"  -> saved {os.path.basename(path)}   "
          f"test AUC (this model alone) = {roc_auc_score(y_test, test_probs_all[-1]):.4f}")

# Averaging the probabilities IS the ensemble.
val_probs_ens  = np.mean(val_probs_all,  axis=0)
test_probs_ens = np.mean(test_probs_all, axis=0)
print("\nAll models trained.")

## 6. Pick the thresholds again

The ensemble produces a different spread of probabilities from the single
baseline model, so its thresholds must be re-tuned — on the **validation fold**,
never on the test fold.

In [ ]:
from sklearn.metrics import precision_recall_curve

def tune_thresholds(y_true, probs, target_recall=0.90):
    prec, rec, thr = precision_recall_curve(y_true, probs)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
    best_f1 = float(thr[np.argmax(f1)])
    ok = rec[:-1] >= target_recall
    high_rec = float(thr[ok][np.argmax(prec[:-1][ok])]) if ok.any() else 0.5
    return best_f1, high_rec


THR_F1, THR_REC = tune_thresholds(y_val, val_probs_ens)
print(f"ensemble best-F1 threshold      : {THR_F1:.4f}")
print(f"ensemble high-recall threshold  : {THR_REC:.4f}")

## 7. Did it actually improve?

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             confusion_matrix)
import pandas as pd

# Score the old baseline model on the same test fold for a fair comparison.
# Its thresholds are re-derived from ITS OWN validation predictions, so both
# models are judged the same way.
baseline_probs = None
if os.path.exists(BASELINE_PATH):
    tf.keras.backend.clear_session()
    _base = tf.keras.models.load_model(BASELINE_PATH)
    baseline_val_probs = _base.predict(X_val,  verbose=0).ravel()
    baseline_probs     = _base.predict(X_test, verbose=0).ravel()
    B_THR_F1, B_THR_REC = tune_thresholds(y_val, baseline_val_probs)
    print(f"baseline thresholds re-derived: best-F1 {B_THR_F1:.4f}, "
          f"high-recall {B_THR_REC:.4f}")


def scores(probs, threshold, label):
    pred = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    return {
        "Model": label,
        "ROC-AUC": roc_auc_score(y_test, probs),
        "PR-AUC": average_precision_score(y_test, probs),
        "Recall": recall_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Specificity": tn / (tn + fp),
        "F1": f1_score(y_test, pred),
        "Accuracy": accuracy_score(y_test, pred),
        "Missed MI": int(fn),
    }


rows = []
if baseline_probs is not None:
    rows.append(scores(baseline_probs, B_THR_REC, "Baseline (no augmentation)"))
for i, p in enumerate(test_probs_all):
    t_f1, t_rec = tune_thresholds(y_val, val_probs_all[i])
    rows.append(scores(p, t_rec, f"Augmented model {i + 1} alone"))
rows.append(scores(test_probs_ens, THR_REC, "ENSEMBLE of 5"))

table = pd.DataFrame(rows).set_index("Model")
print("All rows use each model's own high-recall (>=90% on validation) threshold.\n")
display(table.style.format({c: "{:.4f}" for c in table.columns if c != "Missed MI"}))

### Is the improvement real, or just luck?

The test fold has only 1,462 recordings, so a small rise in AUC can easily be
noise. This resamples the test set 2,000 times and measures the **difference**
between the two models on each resample.

If the 95% interval for the difference **excludes 0**, the improvement is real.
If it includes 0, you cannot honestly claim the ensemble is better — and saying
so in your thesis is a mark of good science, not a failure.

In [ ]:
def bootstrap_difference(y_true, probs_a, probs_b, n_boot=2000, seed=0):
    """95% CI for AUC(b) - AUC(a), using the SAME resample for both models."""
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        if y_true[idx].sum() in (0, len(idx)):
            continue
        diffs.append(roc_auc_score(y_true[idx], probs_b[idx]) -
                     roc_auc_score(y_true[idx], probs_a[idx]))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return float(np.mean(diffs)), float(lo), float(hi)


if baseline_probs is not None:
    d, lo, hi = bootstrap_difference(y_test, baseline_probs, test_probs_ens)
    print(f"Baseline AUC : {roc_auc_score(y_test, baseline_probs):.4f}")
    print(f"Ensemble AUC : {roc_auc_score(y_test, test_probs_ens):.4f}")
    print(f"\nDifference   : {d:+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]")
    print()
    if lo > 0:
        print("VERDICT: the interval excludes 0 - the ensemble is genuinely better.")
    else:
        print("VERDICT: the interval includes 0 - this could be chance. Report it as")
        print("         'no statistically significant difference' rather than a win.")
else:
    print("Baseline model not found in Drive, so no comparison was made.")

In [ ]:
# ROC curves, baseline vs ensemble.
from sklearn.metrics import roc_curve

plt.figure(figsize=(6.5, 6))
if baseline_probs is not None:
    fpr, tpr, _ = roc_curve(y_test, baseline_probs)
    plt.plot(fpr, tpr, lw=2, ls="--", color="#8b949e",
             label=f"Baseline (AUC {roc_auc_score(y_test, baseline_probs):.3f})")
fpr, tpr, _ = roc_curve(y_test, test_probs_ens)
plt.plot(fpr, tpr, lw=2.2, color="#1f6feb",
         label=f"Ensemble (AUC {roc_auc_score(y_test, test_probs_ens):.3f})")
plt.plot([0, 1], [0, 1], "k:", lw=1, label="random guessing")
plt.xlabel("False positive rate  (1 - specificity)"); plt.ylabel("Recall")
plt.title("Baseline vs ensemble - test fold")
plt.legend(loc="lower right"); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "baseline_vs_ensemble_roc.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Save the ensemble for the Flask app

In [ ]:
ens_config = {
    "task": "PTB-XL Lead I, Normal (0) vs Myocardial Infarction (1)",
    "type": "ensemble",
    "members": [f"model_seed{s}.keras" for s in SEEDS],
    "combine": "mean of the 5 predicted probabilities",
    "sample_rate_hz": 100,
    "input_length": int(SIG_LEN),
    "preprocessing": "per-recording z-score: (x - mean) / (std + 1e-8)",
    "augmentation": "train only: time shift +/-1s, baseline wander 0.15-0.5Hz, "
                    "gaussian noise sd<=0.05, then re-z-score",
    "threshold_default": 0.5,
    "threshold_best_f1": round(THR_F1, 4),
    "threshold_high_recall": round(THR_REC, 4),
    "split": {"train": "folds 1-8", "val": "fold 9", "test": "fold 10"},
    "test_roc_auc": round(float(roc_auc_score(y_test, test_probs_ens)), 4),
}
with open(os.path.join(ENSEMBLE_DIR, "ensemble_config.json"), "w") as f:
    json.dump(ens_config, f, indent=2)

table.to_csv(os.path.join(OUT_DIR, "improvement_comparison.csv"))
np.savez_compressed(os.path.join(OUT_DIR, "ensemble_test_predictions.npz"),
                    y_true=y_test, probs=test_probs_ens,
                    member_probs=np.array(test_probs_all))

print("Saved to", ENSEMBLE_DIR)
for f_ in sorted(os.listdir(ENSEMBLE_DIR)):
    print("  ", f_)
print("\nDownload the whole 'ensemble' folder from Drive into your project,")
print("then run the Flask app - it detects the ensemble automatically.")

---

## For Chapter 5

Report the comparison table as a **baseline vs improved** experiment rather than a
single final number — it shows method, not just a result.

Things to state honestly:

- Augmentation and ensembling do **not** add information. They reduce variance and
  over-fitting. The ceiling is still set by what Lead I can physically show, which
  is why inferior and posterior infarctions stay hard.
- If the bootstrap interval for the difference includes 0, say so plainly. "No
  significant difference on a 1,462-record test fold" is a legitimate, publishable
  finding and is far better than over-claiming a 0.005 rise.
- The augmentations were chosen to mimic real electrode artefacts (movement,
  breathing, electrical noise), so this experiment also supports **RQ1** — it is
  evidence about robustness, not only about accuracy.
- The thresholds were re-tuned on the validation fold for the ensemble, and the
  test fold was scored once with them.